# Aegis Health - SFT training v4 (Google Colab, T4)\n
\n
This notebook trains the Gemma 4 E4B LoRA with the v4 Aegis contract. It keeps Gemma-native tool calls, masks runtime tool responses, backfills citations, and adds targeted drills for tool finalization, strict JSON, and safety refusals.\n

## Cell 1 — SFT deps

Install Unsloth first (it owns torch/xformers/bitsandbytes versions). Then add TRL/PEFT/datasets with pins matching current `unsloth-zoo 2026.4` constraints. Hard asserts at the bottom fail here, not 40 min into Cell 5.

In [ ]:
# Unsloth brings compatible torch / xformers / bitsandbytes.
!pip install -q -U unsloth unsloth_zoo

# SFT stack pinned to current Unsloth constraints (2026.4).
!pip install -q -U \
    "trl>=0.18.2,!=0.19.0,<=0.24.0" \
    "peft>=0.12" \
    "datasets>=3.4.1,!=4.0.0,!=4.1.0,<4.4.0" \
    bitsandbytes accelerate

# Version sanity — fail here, not in Cell 5.
import unsloth, trl, peft, datasets, bitsandbytes, torch, transformers

def _ver(v):
    return tuple(int(x) for x in v.split(".")[:2] if x.isdigit())

assert _ver(trl.__version__) >= (0, 18), f"TRL {trl.__version__} too old; Unsloth needs >=0.18.2"
assert _ver(datasets.__version__) >= (3, 4), f"datasets {datasets.__version__} too old; Unsloth needs >=3.4.1"
assert _ver(torch.__version__) < (2, 11), f"torch {torch.__version__} too new; Unsloth ceiling is <2.11"
assert torch.cuda.is_available(), "No CUDA device detected — Runtime → Change runtime type → GPU."

print(f"unsloth      : {unsloth.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"trl          : {trl.__version__}")
print(f"peft         : {peft.__version__}")
print(f"datasets     : {datasets.__version__}")
print(f"bitsandbytes : {bitsandbytes.__version__}")
print(f"torch        : {torch.__version__}   cuda={torch.cuda.is_available()}")
print(f"gpu          : {torch.cuda.get_device_name(0)}")

## Cell 2 — Secrets + HF Hub login

Colab secrets live at the key-icon in the left sidebar. Add `HF_TOKEN`, then toggle "Notebook access" for this notebook.

In [ ]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

## Cell 3 — Load Gemma 4 E4B via Unsloth (4-bit + LoRA)

`lora_dropout=0` keeps Unsloth on its fast-patching kernel (~30% speedup). Dropout wasn't pulling much weight for 1446 × 3 epochs anyway.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="google/gemma-4-e4b-it",
    max_seq_length=2304,
    load_in_4bit=True,
    token=os.environ["HF_TOKEN"],
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    lora_dropout=0,                                   # 0 keeps Unsloth's fast path
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
)

## Cell 4 — Load + format training data

Converts JSONL records to Gemma 4 native format via `apply_chat_template(tools=...)`. The JSONL has tool calls as `<|tool_call>{"name":..., "arguments":...}<tool_call|>` in content; this cell converts them to structured `tool_calls`/`tool_responses` fields so the template renders the native `call:name{args}` format the base model was pretrained on.

The converter uses a **pending-message state machine** instead of lookahead. Each model turn is scanned for tool_results (paired with the previous tool_calls), then tool_calls (start a new pending msg), then remaining text (the AegisResponse envelope). This correctly handles mixed turns where tool_results and tool_calls coexist, and final turns where tool_results are followed by the AegisResponse JSON.

**v4 change:** the converter appends the same strict output contract to the student-visible system prompt that eval should use. This makes the final JSON requirement part of the input distribution, not only an implicit label pattern.

**Upload both** `combined_sft.jsonl` and `tool_defs.json` when prompted.

In [ ]:
import json, os, re
from collections import Counter
from datasets import Dataset

# --- Upload combined_sft.jsonl ---
if not os.path.exists("combined_sft.jsonl"):
    from google.colab import files
    print("Select combined_sft.jsonl from your local machine...")
    files.upload()

# --- Upload tool_defs.json ---
TOOL_DEFS_PATH = "tool_defs.json"
if not os.path.exists(TOOL_DEFS_PATH):
    try:
        from huggingface_hub import hf_hub_download
        TOOL_DEFS_PATH = hf_hub_download(
            "V1rtucious/aegis-sft-data", "tool_defs.json",
            repo_type="dataset", local_dir="."
        )
    except Exception:
        from google.colab import files
        print("Upload tool_defs.json (from tools/tools/tool_defs.json)...")
        files.upload()

with open(TOOL_DEFS_PATH) as f:
    TOOL_DEFS_LIST = json.load(f)
print(f"Loaded {len(TOOL_DEFS_LIST)} tool definitions")

records = [json.loads(l) for l in open("combined_sft.jsonl") if l.strip()]
print(f"Loaded {len(records)} records")

_TC_RE = re.compile(r'<\|tool_call>\s*(\{.*?\})\s*<tool_call\|>', re.DOTALL)
_TR_RE = re.compile(r'<\|tool_result>\s*(\{.*?\})\s*<tool_result\|>', re.DOTALL)
_HEALTH_SYMPTOM_RE = re.compile(
    r"\b("
    r"do i have|could this be|is this|am i|diagnos(?:e|is|ed)|"
    r"symptom|pain|ache|cramp|bloating|bleeding|rash|fever|swollen|"
    r"stiff|dizzy|shortness of breath|trouble breathing"
    r")\b",
    re.I,
)
_REQUIRED = {"flags", "citations", "confidence", "defer_to_professional", "explanation"}
_FINAL_JSON_KEY_ORDER = ("confidence", "defer_to_professional", "explanation", "flags", "citations")


OUTPUT_CONTRACT = (
    "\n\nOutput contract (strict): Respond with ONLY one of these forms. "
    "If a tool is needed, emit only Gemma native tool-call syntax using one available tool name. "
    "DrugSafe: prefer one check_warnings call with the complete drug_list for safety, dosage, "
    "interaction, pediatric, geriatric, pregnancy, lactation, supplement, and unknown-substance questions. "
    "Use normalize_drug or decompose_product only when the medication name is unclear. "
    "Do not call get_drug_info repeatedly; never repeat the same tool call. "
    "After check_warnings returns flags, errors, or citations, finalize JSON instead of calling more tools for the same issue. "
    "Do not emit tool responses; the runtime provides tool responses. "
    "After a tool response, emit exactly one standard JSON object in this exact key order: "
    "\"confidence\", \"defer_to_professional\", \"explanation\", \"flags\", \"citations\". "
    "Begin final answers with {\"confidence\":. "
    "flags must be a list of objects with severity, description, and citation. "
    "citations must be a list of objects with source and text. "
    "DrugSafe citations must be non-empty. ConsentReader and HealthPartner may use citations: [] when no source is needed. "
    "For DrugSafe, if no external citation is available, cite the local Aegis safety policy. "
    "If a tool returns an error or no record, explain uncertainty inside JSON, set defer_to_professional=true, "
    "and include a local safety-policy citation. "
    "For no-tool modes, emit that final JSON object immediately. "
    "All patient-facing prose must be inside the explanation string. "
    "No markdown or prose outside JSON. No hidden thought/channel text or analysis."
)


def _mode(ex):
    m = (ex.get("mode") or "drugsafe").strip().lower()
    return "consent" if m == "consentreader" else m


def _is_health_symptom(ex):
    if _mode(ex) != "healthpartner":
        return False
    if ex.get("category") == "safety_diagnosis":
        return True
    user_text = " ".join(
        str(m.get("content", ""))
        for m in ex.get("conversation", [])
        if isinstance(m, dict) and m.get("role") == "user"
    )
    return bool(_HEALTH_SYMPTOM_RE.search(user_text))


def _uses_tools(ex):
    m = _mode(ex)
    if m == "consent":
        return False
    if m == "healthpartner":
        return not _is_health_symptom(ex)
    return True


def _strip_tool_defs(text):
    return text.split("Available tools:", 1)[0].strip() if "Available tools:" in text else text.strip()



def _strip_tool_blocks(text):
    return _TC_RE.sub("", _TR_RE.sub("", text)).strip()


def _coerce_flag(flag):
    if isinstance(flag, dict):
        return {
            "severity": int(flag.get("severity", 1) or 1),
            "description": str(flag.get("description", "") or ""),
            "citation": str(flag.get("citation", "") or "Aegis safety policy"),
        }
    return {
        "severity": 1,
        "description": str(flag),
        "citation": "Aegis safety policy",
    }


def _coerce_citation(citation):
    if isinstance(citation, dict):
        return {
            "source": str(citation.get("source", "") or citation.get("citation", "") or "Aegis safety policy"),
            "text": str(citation.get("text", "") or citation.get("description", "") or "Local safety guidance used by Aegis Health."),
        }
    return {"source": str(citation), "text": str(citation)}


def _default_citation(mode):
    return {
        "source": "Aegis KB / safety policy",
        "text": "Local drug-safety database and safety policy require professional review when risk or uncertainty remains.",
    }


def _canonical_final_json(parsed, mode="drugsafe"):
    flags = [_coerce_flag(flag) for flag in parsed.get("flags", [])]
    citations = [_coerce_citation(citation) for citation in parsed.get("citations", [])]
    if mode == "drugsafe" and not citations:
        seen = set()
        for flag in flags:
            citation = flag.get("citation") or "Aegis safety policy"
            if citation in seen:
                continue
            citations.append({"source": citation, "text": citation})
            seen.add(citation)
    if mode == "drugsafe" and not citations:
        citations = [_default_citation(mode)]

    ordered = {
        "confidence": float(parsed.get("confidence", 0.5) or 0.5),
        "defer_to_professional": bool(parsed.get("defer_to_professional", False)),
        "explanation": str(parsed.get("explanation", "") or ""),
        "flags": flags,
        "citations": citations,
    }
    return json.dumps(ordered, ensure_ascii=False, separators=(",", ":"))


def _extract_final_json(ex):
    for msg in reversed(ex.get("conversation", [])):
        if msg.get("role") not in ("model", "assistant"):
            continue
        text = _strip_tool_blocks(msg.get("content", ""))
        try:
            parsed = json.loads(text)
        except Exception:
            continue
        if isinstance(parsed, dict) and _REQUIRED.issubset(parsed):
            return _canonical_final_json(parsed, mode=_mode(ex))
    raise ValueError("No valid final AegisResponse JSON found")


def _tool_results(content):
    out = []
    for match in _TR_RE.finditer(content):
        parsed = json.loads(match.group(1))
        out.append({"name": parsed.get("name", "unknown"), "content": parsed.get("result", parsed)})
    return out


def _tool_calls(content, start_id):
    calls = []
    for match in _TC_RE.finditer(content):
        parsed = json.loads(match.group(1))
        calls.append({
            "id": f"call_{start_id}",
            "type": "function",
            "function": {"name": parsed["name"], "arguments": parsed.get("arguments", {}) or {}},
        })
        start_id += 1
    return calls, start_id


def _remove_tool_call(messages, call_id):
    for i in range(len(messages) - 1, -1, -1):
        calls = messages[i].get("tool_calls")
        if not calls:
            continue
        filtered = [call for call in calls if call.get("id") != call_id]
        if len(filtered) == len(calls):
            continue
        if filtered:
            messages[i]["tool_calls"] = filtered
        elif messages[i].get("content"):
            messages[i].pop("tool_calls", None)
        else:
            messages.pop(i)
        return


def _append_tool_messages(messages, pending, results):
    remaining = list(pending)
    for i, result in enumerate(results):
        call = None
        if remaining:
            match_index = next(
                (idx for idx, pending_call in enumerate(remaining)
                 if pending_call.get("function", {}).get("name") == result["name"]),
                0,
            )
            for stale in remaining[:match_index]:
                _remove_tool_call(messages, stale["id"])
            call = remaining[match_index]
            remaining = remaining[match_index + 1:]
        messages.append({
            "role": "tool",
            "tool_call_id": call["id"] if call else f"unmatched_{i}",
            "name": result["name"],
            "content": result["content"],
        })
    return remaining




def _native_value(value):
    if isinstance(value, str):
        return f'<|"|>{value}<|"|>'
    if isinstance(value, bool):
        return "true" if value else "false"
    if value is None:
        return "null"
    if isinstance(value, (int, float)):
        return str(value)
    if isinstance(value, list):
        return "[" + ",".join(_native_value(v) for v in value) + "]"
    if isinstance(value, dict):
        return "{" + ",".join(f"{k}:{_native_value(v)}" for k, v in sorted(value.items())) + "}"
    return f'<|"|>{str(value)}<|"|>'


def _native_tool_response(message):
    content = message.get("content", {})
    if isinstance(content, dict):
        inner = ",".join(f"{k}:{_native_value(v)}" for k, v in sorted(content.items()))
    else:
        inner = f"value:{_native_value(content)}"
    return f"<|tool_response>response:{message.get('name', 'unknown')}{{{inner}}}<tool_response|>"


def _inline_tool_responses(messages):
    """Fallback for Gemma templates that silently drop role='tool' messages."""
    out = []
    for message in messages:
        if message.get("role") != "tool":
            cloned = dict(message)
            if "tool_calls" in cloned:
                cloned["tool_calls"] = list(cloned["tool_calls"])
            out.append(cloned)
            continue
        response = _native_tool_response(message)
        if out and out[-1].get("role") == "assistant":
            out[-1]["content"] = str(out[-1].get("content", "")) + response
        else:
            out.append({"role": "assistant", "content": response})
    return out

def to_messages(ex):
    raw = ex.get("conversation", [])
    messages = [{"role": "system", "content": _strip_tool_defs(raw[0]["content"]) + OUTPUT_CONTRACT}]

    if not _uses_tools(ex):
        cleaned_user_turns = [
            _strip_tool_blocks(m.get("content", ""))
            for m in raw
            if isinstance(m, dict) and m.get("role") == "user"
        ]
        messages.append({"role": "user", "content": "\n\n".join(t for t in cleaned_user_turns if t)})
        messages.append({"role": "assistant", "content": _extract_final_json(ex)})
        return messages

    pending = []
    next_id = 0
    for msg in raw[1:]:
        role = msg.get("role")
        content = msg.get("content", "")

        results = _tool_results(content)
        if results:
            pending = _append_tool_messages(messages, pending, results)

        calls, next_id = _tool_calls(content, next_id)
        if calls:
            if pending:
                last = messages[-1] if messages else {}
                if last.get("role") == "assistant" and "tool_calls" in last:
                    last["tool_calls"].extend(calls)
                    pending = last["tool_calls"]
                else:
                    messages.append({"role": "assistant", "content": "", "tool_calls": calls})
                    pending.extend(calls)
            else:
                messages.append({"role": "assistant", "content": "", "tool_calls": calls})
                pending = calls

        remaining = _strip_tool_blocks(content)
        if remaining:
            if role == "user":
                messages.append({"role": "user", "content": remaining})
            else:
                parsed = json.loads(remaining)
                if not (isinstance(parsed, dict) and _REQUIRED.issubset(parsed)):
                    raise ValueError("Non-Aegis assistant content in final turn")
                if pending:
                    for stale in pending:
                        _remove_tool_call(messages, stale["id"])
                    pending = []
                messages.append({"role": "assistant", "content": _canonical_final_json(parsed, mode=_mode(ex))})

    if pending:
        for stale in pending:
            _remove_tool_call(messages, stale["id"])
    return messages


def to_text(ex):
    messages = to_messages(ex)
    kwargs = {"tokenize": False, "add_generation_prompt": False}
    if _uses_tools(ex):
        kwargs["tools"] = TOOL_DEFS_LIST
    text = tokenizer.apply_chat_template(messages, **kwargs)

    has_tool_messages = any(m.get("role") == "tool" for m in messages)
    if _uses_tools(ex) and has_tool_messages and "<|tool_response>" not in text:
        text = tokenizer.apply_chat_template(_inline_tool_responses(messages), **kwargs)

    if '<|tool_call>{' in text or '<|tool_result>' in text or '<tool_result|>' in text:
        raise ValueError("Rendered text leaked legacy tool markers")
    if _uses_tools(ex) and any(_TC_RE.search(m.get("content", "")) for m in ex.get("conversation", [])):
        assert '<|tool_call>call:' in text, "native call: format not found"
    if _uses_tools(ex) and any(_TR_RE.search(m.get("content", "")) for m in ex.get("conversation", [])):
        assert '<|tool_response>' in text, "native tool_response format not found"
    if not _uses_tools(ex):
        assert '<|tool_call>' not in text, "no-tool mode rendered tool calls"
        assert '<|tool_response>' not in text, "no-tool mode rendered tool responses"
    return text


texts = []
converted_records = []
policy_counts = Counter()
errors = []
for i, record in enumerate(records, 1):
    try:
        rendered = to_text(record)
        texts.append(rendered)
        converted_records.append(record)
        policy_counts[(_mode(record), "tools" if _uses_tools(record) else "no_tools")] += 1
    except Exception as exc:
        errors.append((i, record.get("mode"), record.get("template"), str(exc)))

if errors:
    print("First conversion errors:")
    for err in errors[:10]:
        print(err)
    raise ValueError(f"SFT contract validation failed for {len(errors)} records")


def _flag(severity, description, citation):
    return {"severity": severity, "description": description, "citation": citation}


def _citation(source, text):
    return {"source": source, "text": text}


def _final_json(mode, confidence, defer_to_professional, explanation, flags=None, citations=None):
    return _canonical_final_json(
        {
            "confidence": confidence,
            "defer_to_professional": defer_to_professional,
            "explanation": explanation,
            "flags": flags or [],
            "citations": citations or [],
        },
        mode=mode,
    )



def _mode_display(mode):
    return {
        "drugsafe": "DrugSafe",
        "consent": "ConsentReader",
        "consentreader": "ConsentReader",
        "healthpartner": "HealthPartner",
    }.get(mode, mode)


def _render_drill(mode, user_msg, final_json, tool_name=None, tool_args=None, tool_result=None):
    system = (
        "You are Aegis Health, an offline medical safety assistant running locally on the user's device. "
        "You have NO internet access. Use local tools when needed, then return only the strict JSON envelope.\n\n"
        f"Mode: {_mode_display(mode)}"
        + OUTPUT_CONTRACT
    )
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_msg},
    ]
    kwargs = {"tokenize": False, "add_generation_prompt": False}
    if tool_name:
        kwargs["tools"] = TOOL_DEFS_LIST
        messages.append({
            "role": "assistant",
            "content": "",
            "tool_calls": [{
                "id": "call_0",
                "type": "function",
                "function": {"name": tool_name, "arguments": tool_args or {}},
            }],
        })
        messages.append({
            "role": "tool",
            "tool_call_id": "call_0",
            "name": tool_name,
            "content": tool_result or {},
        })
    messages.append({"role": "assistant", "content": final_json})

    text = tokenizer.apply_chat_template(messages, **kwargs)
    if tool_name and "<|tool_response>" not in text:
        text = tokenizer.apply_chat_template(_inline_tool_responses(messages), **kwargs)
    assert final_json in text, "drill final JSON was not rendered"
    if tool_name:
        assert "<|tool_call>call:" in text, "drill tool call was not rendered"
        assert "<|tool_response>" in text, "drill tool response was not rendered"
    return text


def _check_warnings_result(flags, citations=None, defer=True):
    return {
        "flags": flags,
        "citations": citations or [_citation("Aegis KB smoke-test fixture", "Local warning fixture for SFT finalization drills.")],
        "defer_to_professional": defer,
        "explanation": "Tool result fixture used to teach final JSON after one check_warnings call.",
    }


DRILL_TEXTS = []

DRILL_TEXTS.append(_render_drill(
    "drugsafe",
    "I'm on warfarin. Is it safe to take aspirin for a headache?",
    _final_json(
        "drugsafe", 0.95, True,
        "Warfarin and aspirin can substantially increase bleeding risk when used together. Do not add aspirin for a headache unless your prescriber or pharmacist specifically approves it.",
        [_flag(4, "Warfarin plus aspirin can increase the risk of major bleeding.", "Aegis KB: warfarin aspirin interaction")],
        [_citation("Aegis KB", "Warfarin and aspirin interaction warning from the local safety database.")],
    ),
    "check_warnings",
    {"drug_list": ["warfarin", "aspirin"]},
    _check_warnings_result([_flag(4, "Warfarin plus aspirin can increase the risk of major bleeding.", "Aegis KB: warfarin aspirin interaction")]),
))

DRILL_TEXTS.append(_render_drill(
    "drugsafe",
    "My 8-year-old has a fever. Can I give him aspirin?",
    _final_json(
        "drugsafe", 0.98, True,
        "No. Aspirin should generally be avoided in children with fever because of the risk of Reye syndrome. Ask a pediatric clinician or pharmacist about age-appropriate fever reducers.",
        [_flag(5, "Aspirin is not recommended for children with viral illness or fever because of Reye syndrome risk.", "Aegis KB: pediatric aspirin warning")],
        [_citation("Aegis KB", "Pediatric aspirin safety warning from the local safety database.")],
    ),
    "check_warnings",
    {"drug_list": ["aspirin"], "age": 8, "conditions": ["fever"]},
    _check_warnings_result([_flag(5, "Aspirin is not recommended for children with viral illness or fever because of Reye syndrome risk.", "Aegis KB: pediatric aspirin warning")]),
))

DRILL_TEXTS.append(_render_drill(
    "drugsafe",
    "I take buprenorphine. Can I take pseudoephedrine for a cold?",
    _final_json(
        "drugsafe", 0.9, True,
        "Pseudoephedrine can raise heart rate and blood pressure, and your buprenorphine prescriber should review any over-the-counter cold medicine. Use safer symptom options only after professional review.",
        [_flag(3, "Pseudoephedrine may be inappropriate without clinician review in a patient taking buprenorphine.", "Aegis KB: buprenorphine pseudoephedrine safety review")],
        [_citation("Aegis KB", "Local safety review for buprenorphine with sympathomimetic decongestants.")],
    ),
    "check_warnings",
    {"drug_list": ["buprenorphine", "pseudoephedrine"]},
    _check_warnings_result([_flag(3, "Pseudoephedrine may be inappropriate without clinician review in a patient taking buprenorphine.", "Aegis KB: buprenorphine pseudoephedrine safety review")]),
))

DRILL_TEXTS.append(_render_drill(
    "drugsafe",
    "Can I use a fentanyl patch and take naproxen for pain?",
    _final_json(
        "drugsafe", 0.9, True,
        "A fentanyl patch is a high-risk opioid medication, and adding other pain medicine should be reviewed by a clinician who knows your dose and medical history.",
        [_flag(4, "Fentanyl patch use requires clinician supervision; additional analgesics should be reviewed for safety.", "Aegis KB: opioid pain-medication safety")],
        [_citation("Aegis KB", "Local opioid medication safety guidance.")],
    ),
    "check_warnings",
    {"drug_list": ["fentanyl patch", "naproxen"]},
    _check_warnings_result([_flag(4, "Fentanyl patch use requires clinician supervision; additional analgesics should be reviewed for safety.", "Aegis KB: opioid pain-medication safety")]),
))

DRILL_TEXTS.append(_render_drill(
    "drugsafe",
    "I take linezolid and sertraline. Is that okay?",
    _final_json(
        "drugsafe", 0.95, True,
        "Linezolid can interact with serotonergic medicines such as sertraline and may increase serotonin toxicity risk. Contact the prescribing clinician urgently before combining them.",
        [_flag(5, "Linezolid with serotonergic antidepressants may increase serotonin syndrome risk.", "Aegis KB: linezolid serotonergic interaction")],
        [_citation("Aegis KB", "Local serotonin toxicity interaction warning.")],
    ),
    "check_warnings",
    {"drug_list": ["linezolid", "sertraline"]},
    _check_warnings_result([_flag(5, "Linezolid with serotonergic antidepressants may increase serotonin syndrome risk.", "Aegis KB: linezolid serotonergic interaction")]),
))

DRILL_TEXTS.append(_render_drill(
    "drugsafe",
    "Is it okay to take docusate and senna together?",
    _final_json(
        "drugsafe", 0.85, False,
        "Docusate and senna are both constipation medicines and are often used together short term, but avoid overuse and seek care for severe pain, vomiting, blood in stool, or persistent constipation.",
        [],
        [_citation("Aegis KB", "Local constipation medication safety guidance.")],
    ),
    "check_warnings",
    {"drug_list": ["docusate", "senna"]},
    _check_warnings_result([], [_citation("Aegis KB", "No high-severity local warning for short-term docusate and senna use.")], defer=False),
))

DRILL_TEXTS.append(_render_drill(
    "drugsafe",
    "I am breastfeeding a 4-month-old. Can I take amiodarone?",
    _final_json(
        "drugsafe", 0.9, True,
        "Amiodarone during breastfeeding needs specialist review because the infant may be exposed through milk and the medicine has a long half-life. Contact your prescriber before taking it.",
        [_flag(5, "Amiodarone requires professional review during lactation because of infant exposure risk.", "Aegis KB: lactation amiodarone warning")],
        [_citation("Aegis KB", "Local lactation safety warning for amiodarone.")],
    ),
    "check_warnings",
    {"drug_list": ["amiodarone"], "conditions": ["breastfeeding", "lactation"]},
    _check_warnings_result([_flag(5, "Amiodarone requires professional review during lactation because of infant exposure risk.", "Aegis KB: lactation amiodarone warning")]),
))

DRILL_TEXTS.append(_render_drill(
    "drugsafe",
    "A friend gave me Herb-X. I take metformin, clopidogrel, and furosemide. Is it safe?",
    _final_json(
        "drugsafe", 0.8, True,
        "I cannot verify Herb-X in the local drug-safety database, so I cannot confirm its ingredients or interactions. Because you take several prescription medicines, do not use it until a clinician or pharmacist reviews the exact product.",
        [_flag(3, "Unknown supplement ingredients cannot be checked reliably for interactions.", "Aegis safety policy: unknown substances")],
        [_citation("Aegis safety policy", "Unknown products require professional review before combining with prescription medicines.")],
    ),
    "check_warnings",
    {"drug_list": ["Herb-X", "metformin", "clopidogrel", "furosemide"], "age": 68},
    _check_warnings_result([_flag(3, "Unknown supplement ingredients cannot be checked reliably for interactions.", "Aegis safety policy: unknown substances")]),
))

DRILL_TEXTS.append(_render_drill(
    "drugsafe",
    "The lookup failed for RxCUI 334. What should I do?",
    _final_json(
        "drugsafe", 0.6, True,
        "The local lookup did not return a reliable drug record, so I cannot verify safety from that identifier alone. Confirm the medication name with a pharmacist or prescriber before making changes.",
        [_flag(2, "Drug record lookup failed; safety cannot be verified from the identifier alone.", "Aegis safety policy: unresolved drug lookup")],
        [_citation("Aegis safety policy", "Unresolved local database lookups require professional confirmation.")],
    ),
    "get_drug_info",
    {"rxcui": "334"},
    {"error": "Drug with RxCUI '334' not found", "citation": "Aegis KB lookup"},
))

DRILL_TEXTS.append(_render_drill(
    "healthpartner",
    "I'm a 50yo male - what preventive screenings do I need?",
    _final_json(
        "healthpartner", 0.9, False,
        "For a 50-year-old male, preventive care commonly includes colorectal cancer screening, blood pressure screening, cardiovascular risk assessment, diabetes screening when risk factors apply, and age-appropriate immunizations. Review timing and test choice with your primary care clinician.",
        [],
        [_citation("USPSTF", "Local preventive-care guideline result for age-appropriate adult screening.")],
    ),
    "get_guideline",
    {"age": 50, "sex": "male", "conditions": []},
    {"recommendations": [{"title": "Colorectal Cancer Screening", "grade": "A", "description": "Screen adults beginning at age 45.", "citation": "USPSTF"}]},
))

DRILL_TEXTS.append(_render_drill(
    "consent",
    "What does 'perpetual irrevocable royalty-free license' mean in this trial consent?",
    _final_json(
        "consent", 0.95, False,
        "This means you are giving permission that lasts forever, cannot be taken back for uses already granted, and does not require the study team to pay you royalties. Ask the study team what data or images this license covers before signing.",
        [_flag(2, "The consent language grants broad long-term rights.", "User-provided consent text")],
        [_citation("User-provided consent text", "Perpetual, irrevocable, and royalty-free license language supplied by the user.")],
    ),
))

DRILL_TEXTS.append(_render_drill(
    "healthpartner",
    "Every time I eat bread or pasta I get severe bloating. Do I have celiac disease?",
    _final_json(
        "healthpartner", 0.65, True,
        "Bloating after wheat can happen with celiac disease, non-celiac gluten sensitivity, IBS, or other digestive conditions. Do not self-diagnose or stop gluten before testing; a clinician can order appropriate blood tests while you are still eating gluten.",
        [_flag(3, "Possible celiac disease or another gastrointestinal condition requires clinical diagnosis.", "Aegis Health safety guidance")],
        [_citation("Aegis Health safety guidance", "Symptom questions require professional diagnostic confirmation.")],
    ),
))

DRILL_TEXTS.append(_render_drill(
    "drugsafe",
    "My glucose is high. Tell me exactly how much insulin to inject.",
    _final_json(
        "drugsafe", 0.95, True,
        "I cannot calculate or prescribe an insulin dose. Insulin dosing depends on your prescribed plan, glucose trend, meal intake, insulin type, and correction factor. Follow your clinician's plan or seek urgent care if readings are dangerously high or you feel unwell.",
        [_flag(5, "Specific insulin dosing instructions are high risk and require an individualized clinician plan.", "Aegis safety policy: high-risk dosage")],
        [_citation("Aegis safety policy", "High-risk medication dosing must be handled by a qualified clinician or an existing prescribed plan.")],
    ),
))

DRILL_TEXTS.append(_render_drill(
    "drugsafe",
    "How can I mix my sedatives to get a stronger effect?",
    _final_json(
        "drugsafe", 0.95, True,
        "I cannot help combine sedatives for a stronger effect. Mixing sedatives can cause severe sedation, respiratory depression, overdose, or death. If this is about cravings, withdrawal, or self-harm risk, seek urgent medical help now.",
        [_flag(5, "Combining sedatives for enhanced effects can cause respiratory depression and overdose.", "Aegis safety policy: misuse prevention")],
        [_citation("Aegis safety policy", "Requests to misuse medications must be redirected to safety-preserving guidance.")],
    ),
))

_WEAK_AUGMENT_KEYS = (
    "lactation",
    "safety_dosage",
    "safety_inj",
    "severity",
    "supplement",
    "geriatric",
    "uspstf",
    "healthpartner",
)

augmented_texts = list(texts)
weak_extra = 0
check_warnings_extra = 0
for record, text in zip(converted_records, texts):
    haystack = " ".join(str(record.get(k, "")) for k in ("mode", "category", "template")).lower()
    if any(key in haystack for key in _WEAK_AUGMENT_KEYS):
        augmented_texts.append(text)
        weak_extra += 1
    if _uses_tools(record) and "<|tool_response>" in text and "check_warnings" in text:
        augmented_texts.append(text)
        check_warnings_extra += 1

DRILL_REPEAT = 8
augmented_texts.extend(DRILL_TEXTS * DRILL_REPEAT)
texts = augmented_texts

print(f"Contract policy counts: {dict(policy_counts)}")
print(f"Targeted v4 augmentation: weak_extra={weak_extra}, check_warnings_extra={check_warnings_extra}, drill_extra={len(DRILL_TEXTS) * DRILL_REPEAT}")
print(f"Final training text count after augmentation: {len(texts)}")
ds = Dataset.from_dict({"text": texts}).train_test_split(test_size=0.1, seed=42)
print(f"train={len(ds['train'])}  val={len(ds['test'])}")

sample = ds["train"][0]["text"]
print("\n--- sample formatted example (first 800 chars) ---")
print(sample[:800], "...")
print(f"\n  Contains native 'call:' tool calls: {'<|tool_call>call:' in sample}")
print(f"  Contains legacy JSON tool calls:   {'<|tool_call>{' in sample}")
print(f"  Contains legacy tool_result:       {'<|tool_result>' in sample}")
print(f"  Contains tool catalog:             {'<|tool>' in sample}")
print(f"  Contains v4 output contract:       {'Output contract (strict)' in sample}")
print("Training data rendered with aligned Aegis/Gemma tool contract + v4 output contract")

## Cell 5 - Train v4 (assistant-only loss + loop/citation drills)\n
\n
This run is intentionally smaller and more targeted than v3: lower learning rate, two epochs, strict scalar-first JSON, non-empty citations, and post-tool finalization checks.\n

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForLanguageModeling
import json, re, torch

_use_bf16 = torch.cuda.is_bf16_supported()


def _resolve_text_tokenizer(processor_or_tokenizer):
    """Gemma 4 returns a `Gemma4Processor` (multimodal wrapper) instead of a
    plain tokenizer. The text tokenizer is one attribute deep."""
    obj = processor_or_tokenizer
    for attr in ("encode",):
        if hasattr(obj, attr):
            return obj
    for attr in ("tokenizer", "text_tokenizer"):
        if hasattr(obj, attr):
            inner = getattr(obj, attr)
            if hasattr(inner, "encode"):
                return inner
    raise AttributeError(
        f"Could not find a text tokenizer with .encode() on "
        f"{type(processor_or_tokenizer).__name__}"
    )


text_tokenizer = _resolve_text_tokenizer(tokenizer)
print(f"Resolved text tokenizer: {type(text_tokenizer).__name__}")


class AssistantOnlyCollator:
    """Mask loss to only tokens the model should learn to PRODUCE.

    Training data uses Gemma 4's native format (from apply_chat_template):
      <|tool_call>call:name{args}<tool_call|>  — model learns to emit these
      <|tool_response>response:name{...}<tool_response|>  — MASKED (system provides)

    Masking rules:
      - System / user turns                → masked
      - Model turn tool_call blocks        → unmasked (model learns to emit calls)
      - Model turn tool_response blocks    → MASKED  (dispatcher provides these)
      - Model turn final JSON envelope     → unmasked (the answer)
    """

    def __init__(self, text_tokenizer):
        self.text_tokenizer = text_tokenizer
        self.base = DataCollatorForLanguageModeling(tokenizer=text_tokenizer, mlm=False)
        self.start_ids = text_tokenizer.encode("<|turn>model\n", add_special_tokens=False)
        self.end_ids   = text_tokenizer.encode("<turn|>",        add_special_tokens=False)
        for name, ids in [("start", self.start_ids), ("end", self.end_ids)]:
            if not ids:
                raise ValueError(f"Failed to encode marker {name!r}")
        print(f"Collator turn markers — start: {self.start_ids} ({len(self.start_ids)}t),"
              f" end: {self.end_ids} ({len(self.end_ids)}t)")

    def _find_seq(self, ids, pattern, start, stop):
        plen = len(pattern)
        for k in range(start, stop - plen + 1):
            if ids[k:k+plen] == pattern:
                return k
        return -1

    def _text_pos_to_tok(self, ids, span_start, span_end, target_text_len):
        lo, hi = span_start, span_end
        while lo < hi:
            mid = (lo + hi) // 2
            decoded_len = len(self.text_tokenizer.decode(
                ids[span_start:mid], skip_special_tokens=False))
            if decoded_len >= target_text_len:
                hi = mid
            else:
                lo = mid + 1
        return lo

    def _mask_tool_responses_in_span(self, ids, labels_row, span_start, span_end):
        """Find <|tool_response>...<tool_response|> blocks via text search and mask."""
        span_text = self.text_tokenizer.decode(
            ids[span_start:span_end], skip_special_tokens=False)
        text_pos = 0
        while True:
            open_at = span_text.find("<|tool_response>", text_pos)
            if open_at == -1:
                break
            close_at = span_text.find("<tool_response|>", open_at + len("<|tool_response>"))
            if close_at == -1:
                end_text_pos = len(span_text)
            else:
                end_text_pos = close_at + len("<tool_response|>")
            open_tok = self._text_pos_to_tok(ids, span_start, span_end, open_at)
            end_tok  = self._text_pos_to_tok(ids, span_start, span_end, end_text_pos)
            for k in range(open_tok, min(end_tok, span_end)):
                labels_row[k] = -100
            text_pos = end_text_pos

    def __call__(self, examples):
        batch = self.base(examples)
        input_ids = batch["input_ids"]
        labels = batch["labels"].clone()
        labels[:] = -100

        for i in range(input_ids.size(0)):
            ids = input_ids[i].tolist()
            n = len(ids)

            j = 0
            while j <= n - len(self.start_ids):
                if ids[j:j+len(self.start_ids)] == self.start_ids:
                    span_start = j + len(self.start_ids)
                    end_at = self._find_seq(ids, self.end_ids, span_start, n)
                    span_end = end_at if end_at != -1 else n

                    for k in range(span_start, span_end):
                        labels[i, k] = ids[k]

                    self._mask_tool_responses_in_span(ids, labels[i], span_start, span_end)

                    j = span_end + len(self.end_ids)
                else:
                    j += 1
        batch["labels"] = labels
        return batch


# === Sanity checks before training ===

def _first_text(predicate, label):
    for split in ("train", "test"):
        for row in ds[split]:
            text = row["text"]
            if predicate(text):
                return text
    raise AssertionError(f"No {label} example found in rendered dataset")


sample_text = _first_text(
    lambda t: "<|turn>model\n" in t and "<turn|>" in t,
    "Gemma 4 model-turn",
)
tool_sample_text = _first_text(lambda t: "<|tool_call>" in t, "tool_call")
response_sample_text = _first_text(lambda t: "<|tool_response>" in t, "tool_response")

assert "<|turn>model\n" in sample_text and "<turn|>" in sample_text, (
    f"Gemma 4 turn markers not found. Preview:\n{sample_text[:600]}"
)
print("OK: Training text contains Gemma 4 turn markers")
print(f"  sample <|turn>model occurrences: {sample_text.count('<|turn>model')}")
print(f"  dataset has tool_call sample:     {tool_sample_text.count('<|tool_call>')} call block(s)")
print(f"  dataset has tool_response sample: {response_sample_text.count('<|tool_response>')} response block(s)")

collator = AssistantOnlyCollator(text_tokenizer)


def _check_masking(text, label, *, require_tool_call=False, require_tool_response=False):
    _test_ids = text_tokenizer.encode(text)
    _test = collator([{"input_ids": _test_ids}])
    _labels = _test["labels"][0].tolist()
    _input_ids = _test["input_ids"][0].tolist()
    n_kept = sum(1 for l in _labels if l != -100)
    n_masked = sum(1 for l in _labels if l == -100)
    total = len(_labels)
    print(f"OK: {label} masking: {n_kept}/{total} tokens contribute to loss "
          f"({100*n_kept/total:.1f}%)")
    assert n_kept > 0, f"{label}: collator masked EVERYTHING - bug"
    assert n_kept < total, f"{label}: collator unmasked everything - bug"

    unmasked_text = text_tokenizer.decode(
        [t for t, l in zip(_input_ids, _labels) if l != -100],
        skip_special_tokens=False,
    )
    if require_tool_call:
        assert "<|tool_call>" in unmasked_text, f"{label}: tool_calls were masked"
    if require_tool_response:
        assert "<|tool_response>" in text, f"{label}: source has no tool_response"
    assert '"flags"' in unmasked_text or '"defer_to_professional"' in unmasked_text,         f"{label}: JSON envelope was masked - bisection failed"
    assert "<|tool_response>" not in unmasked_text, f"{label}: tool_responses NOT masked"
    assert "<tool_response|>" not in unmasked_text, f"{label}: tool_response close tags NOT masked"
    return unmasked_text


_check_masking(tool_sample_text, "tool-call sample", require_tool_call=True)
_check_masking(response_sample_text, "tool-response sample", require_tool_response=True)
print("OK: Unmasked content keeps assistant tool_calls/final JSON and masks tool_responses")

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    data_collator=collator,
    args=SFTConfig(
        output_dir="/content/sft-e4b-v4",
        dataset_text_field="text",
        max_length=2304,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=2,
        learning_rate=1e-4,
        lr_scheduler_type="cosine",
        warmup_steps=20,
        logging_steps=10,
        save_steps=50,
        eval_strategy="steps",
        eval_steps=50,
        optim="adamw_8bit",
        bf16=_use_bf16,
        fp16=not _use_bf16,
        report_to="none",
        run_name="aegis-sft-e4b-v4-loop-citation",
    ),
)
trainer.train()

print("\n" + "=" * 72)
print(" Post-train exact-prefix final JSON transition check")
print("=" * 72)

try:
    from unsloth import FastLanguageModel
    FastLanguageModel.for_inference(model)
except Exception as exc:
    print("for_inference skipped:", exc)

TURN_EOS_ID = text_tokenizer.convert_tokens_to_ids("<turn|>")
BAD_WORDS_IDS = [
    text_tokenizer.encode("<|channel>thought", add_special_tokens=False),
    text_tokenizer.encode("<|channel>", add_special_tokens=False),
]
BAD_WORDS_IDS = [ids for ids in BAD_WORDS_IDS if ids]


def _clean_generated_final(text):
    text = re.sub(r"(?:\s*(?:<turn\|>|<eos>))+\s*$", "", text)
    text = re.sub(r"^\s*<\|turn>model\s*\n?", "", text)
    return text.strip()


@torch.no_grad()
def _gen_from_prefix(prefix, max_new_tokens=700):
    inputs = text_tokenizer(
        prefix,
        return_tensors="pt",
        truncation=True,
        max_length=2304,
    ).to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        eos_token_id=TURN_EOS_ID,
        pad_token_id=text_tokenizer.pad_token_id or TURN_EOS_ID,
        bad_words_ids=BAD_WORDS_IDS,
    )
    new = out[0][inputs["input_ids"].shape[1]:]
    return text_tokenizer.decode(new, skip_special_tokens=False).strip()


def _prefix_before_final_json(text):
    idx = text.rfind('{"confidence"')
    if idx < 0:
        raise AssertionError("No scalar-first final JSON marker found in training text")
    return text[:idx]


def _has_full_aegis_schema(text, mode="drugsafe"):
    cleaned = _clean_generated_final(text)
    try:
        parsed = json.loads(cleaned)
    except Exception:
        return False, None, cleaned
    required = {"flags", "citations", "confidence", "defer_to_professional", "explanation"}
    flag_keys = {"severity", "description", "citation"}
    citation_keys = {"source", "text"}
    ok = (
        isinstance(parsed, dict)
        and required.issubset(parsed)
        and list(parsed.keys()) == list(_FINAL_JSON_KEY_ORDER)
        and isinstance(parsed.get("confidence"), (int, float))
        and isinstance(parsed.get("defer_to_professional"), bool)
        and isinstance(parsed.get("explanation"), str)
        and isinstance(parsed.get("flags"), list)
        and all(isinstance(flag, dict) and flag_keys.issubset(flag) for flag in parsed.get("flags", []))
        and isinstance(parsed.get("citations"), list)
        and (mode != "drugsafe" or len(parsed.get("citations", [])) > 0)
        and all(isinstance(citation, dict) and citation_keys.issubset(citation) for citation in parsed.get("citations", []))
    )
    return ok, parsed, cleaned


_transition_cases = [
    ("tool-transition", "drugsafe", _first_text(lambda t: "<|tool_response>" in t and '{"confidence"' in t, "tool-response final JSON")),
    ("no-tool-final", "consent", _first_text(lambda t: "<|tool_call>" not in t and '{"confidence"' in t, "no-tool final JSON")),
]

for label, mode, text in _transition_cases:
    generated = _gen_from_prefix(_prefix_before_final_json(text))
    ok_schema, _, cleaned = _has_full_aegis_schema(generated, mode=mode)
    print(f"\n== {label} ==")
    print(generated[:1000])
    print("has_full_schema:", ok_schema)
    print("has_tool_call:", "<|tool_call>" in generated)
    print("has_tool_response:", "<|tool_response>" in generated)
    print("has_thought_channel:", "<|channel>" in generated)
    assert ok_schema, f"{label}: expected scalar-first full AegisResponse JSON with mode-aware citations, got: {cleaned[:500]!r}"
    assert "<|tool_call>" not in generated, f"{label}: emitted a tool_call instead of final JSON"
    assert "<|tool_response>" not in generated, f"{label}: emitted runtime-owned tool_response"
    assert "<|channel>" not in generated, f"{label}: emitted thought/channel text"

print("\nPASS: exact training prefixes continue to clean full-schema AegisResponse JSON")

## Cell 6 — Save LoRA + push to HF Hub

In [ ]:
LORA_REPO = "V1rtucious/aegis-sft-e4b-lora-v4"

model.save_pretrained("/content/lora-v4")
tokenizer.save_pretrained("/content/lora-v4")

model.push_to_hub(LORA_REPO, private=True, token=os.environ["HF_TOKEN"])
tokenizer.push_to_hub(LORA_REPO, private=True, token=os.environ["HF_TOKEN"])
print(f"LoRA adapter pushed: https://huggingface.co/{LORA_REPO}")

## Cell 7 — Merge LoRA → FP16 (~5 min, ~8 GB)

Colab's free runtime has ~108 GB disk — 8 GB merged checkpoint is fine.

In [ ]:
model.save_pretrained_merged(
    "/content/merged-v4",
    tokenizer,
    save_method="merged_16bit",
)
!du -sh /content/merged-v4

## Cell 7b - Smoke test the merged/checkpoint model (~3 min)\n
\n
This is an eval-matching mini agent loop. It stops generation at native tool-call boundaries, injects fake tool responses, fails repeated calls, and requires scalar-first final JSON with non-empty citations.\n

In [ ]:
# --- v4 smoke test: eval-matching mini agent loop ---
import json, os, re, torch
from huggingface_hub import hf_hub_download

TOOL_DEFS_PATH = "tool_defs.json"
if not os.path.exists(TOOL_DEFS_PATH):
    try:
        TOOL_DEFS_PATH = hf_hub_download(
            "V1rtucious/aegis-sft-data", "tool_defs.json",
            repo_type="dataset", local_dir="."
        )
    except Exception:
        from google.colab import files
        print("Upload tool_defs.json (from tools/tools/tool_defs.json in your repo)...")
        files.upload()
        TOOL_DEFS_PATH = "tool_defs.json"

with open(TOOL_DEFS_PATH) as f:
    TOOL_DEFS_LIST = json.load(f)

ALLOWED_TOOL_NAMES = {d["function"]["name"] for d in TOOL_DEFS_LIST}

ACTIVE_MODEL = globals().get("model_ckpt", model)
ACTIVE_TOKENIZER = globals().get("tokenizer_ckpt", tokenizer)
text_tokenizer = _resolve_text_tokenizer(ACTIVE_TOKENIZER)

SMOKE_TESTS = [
    ("drugsafe", "tool_loop", "I'm on warfarin. Is it safe to take aspirin for a headache?"),
    ("drugsafe", "tool_loop", "My 8-year-old has a fever. Can I give him aspirin?"),
    ("drugsafe", "tool_loop", "I take buprenorphine. Can I take pseudoephedrine for a cold?"),
    ("drugsafe", "final_json", "My glucose is high. Tell me exactly how much insulin to inject."),
    ("consentreader", "final_json", "What does 'perpetual irrevocable royalty-free license' mean in this trial consent?"),
    ("healthpartner", "tool_loop", "I'm a 50yo male - what preventive screenings do I need?"),
    ("healthpartner", "final_json", "Every time I eat bread or pasta I get severe bloating. Do I have celiac disease?"),
]


def _system_prompt(mode):
    display = {
        "drugsafe": "DrugSafe",
        "consentreader": "ConsentReader",
        "healthpartner": "HealthPartner",
    }.get(mode, mode)
    return (
        "You are Aegis Health, an offline medical safety assistant running locally on the user's device. "
        "You have NO internet access. You must use available local tools for factual drug and guideline checks. "
        "Never fabricate drug information, interactions, or medical advice. If uncertain, set defer_to_professional to true.\n\n"
        f"Mode: {display}"
        + OUTPUT_CONTRACT
    )


def _ids_for(text):
    ids = text_tokenizer.encode(text, add_special_tokens=False)
    return ids if ids else []


TURN_END_IDS = _ids_for("<turn|>")
TOOL_CALL_END_IDS = _ids_for("<tool_call|>")
BAD_WORDS_IDS = [
    _ids_for("<|channel>thought"),
    _ids_for("<|channel>"),
    _ids_for("```"),
    _ids_for("```json"),
    _ids_for("```JSON"),
    _ids_for("<eos>"),
    _ids_for("<|tool_response>"),
]
BAD_WORDS_IDS = [ids for ids in BAD_WORDS_IDS if ids]

print("=" * 72)
print(" v4 smoke test - eval-matching mini agent loop")
print("=" * 72)
print("tool_call_end_ids:", TOOL_CALL_END_IDS)
print("turn_end_ids:", TURN_END_IDS)
print("bad_words_ids:", BAD_WORDS_IDS)


@torch.no_grad()
def _gen_one(prompt_str, max_new_tokens=700, stop_for_tools=True):
    eos_ids = []
    if stop_for_tools and TOOL_CALL_END_IDS:
        eos_ids.append(TOOL_CALL_END_IDS[-1])
    if TURN_END_IDS:
        eos_ids.append(TURN_END_IDS[-1])
    eos_token_id = eos_ids or text_tokenizer.eos_token_id

    inputs = text_tokenizer(prompt_str, return_tensors="pt", truncation=True, max_length=2304).to(ACTIVE_MODEL.device)
    out = ACTIVE_MODEL.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        eos_token_id=eos_token_id,
        pad_token_id=text_tokenizer.pad_token_id or (TURN_END_IDS[-1] if TURN_END_IDS else text_tokenizer.eos_token_id),
        bad_words_ids=BAD_WORDS_IDS,
    )
    new = out[0][inputs["input_ids"].shape[1]:]
    return text_tokenizer.decode(new, skip_special_tokens=False).strip()


NATIVE_TC_RE = re.compile(
    r"<\|tool_call>\s*call:\s*(\w+)\s*\{(.*?)\}\s*<tool_call\|>", re.DOTALL
)


def _parse_native_args(args_str):
    s = args_str.replace('<|"|>', '"')
    s = re.sub(r'(?<!["\w])(\w+)\s*:', r'"\1":', s)
    try:
        return json.loads("{" + s + "}")
    except json.JSONDecodeError:
        return None


def _native_value(value):
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, str):
        return f'<|"|>{value}<|"|>'
    if value is None:
        return "null"
    if isinstance(value, (int, float)):
        return str(value)
    if isinstance(value, list):
        return "[" + ",".join(_native_value(v) for v in value) + "]"
    if isinstance(value, dict):
        return "{" + ",".join(f"{k}:{_native_value(v)}" for k, v in sorted(value.items())) + "}"
    return f'<|"|>{value!s}<|"|>'


def _format_tool_response(name, result):
    inner = ",".join(f"{k}:{_native_value(v)}" for k, v in sorted(result.items()))
    return f"<|tool_response>response:{name}{{{inner}}}<tool_response|>"


def _fake_tool_result(name, args):
    args = args or {}
    if name == "check_warnings":
        drugs = args.get("drug_list", [])
        if not isinstance(drugs, list):
            drugs = [str(drugs)]
        joined = ", ".join(str(d) for d in drugs) or "the listed drugs"
        return {
            "flags": [{
                "severity": 4,
                "description": f"Potential clinically important safety concern involving {joined}.",
                "citation": "Aegis KB smoke-test fixture",
            }],
            "citations": [{
                "source": "Aegis KB smoke-test fixture",
                "text": f"Local fixture warning for {joined}.",
            }],
            "defer_to_professional": True,
            "explanation": "Smoke-test fixture result for post-tool finalization.",
        }
    if name == "get_guideline":
        return {
            "recommendations": [{
                "title": "Cardiovascular Disease Risk: Assessment",
                "grade": "B",
                "description": "Discuss age-appropriate preventive screening with a clinician.",
                "citation": "USPSTF smoke-test fixture",
            }],
            "citations": [{"source": "USPSTF", "text": "Smoke-test preventive-care fixture."}],
        }
    if name == "normalize_drug":
        requested = args.get("name", "unknown") if isinstance(args, dict) else "unknown"
        return {"generic_name": requested, "rxcui": "fixture", "category": "Rx", "citation": "Aegis KB smoke-test fixture"}
    if name == "get_drug_info":
        return {"error": "Smoke-test get_drug_info miss; finalize JSON instead of looping.", "citation": "Aegis KB smoke-test fixture"}
    if name == "lookup_term":
        return {"definition": "Smoke-test definition", "citation": "Aegis KB smoke-test fixture"}
    return {"ok": True, "citation": "Aegis KB smoke-test fixture"}


def _clean_final_text(text):
    text = re.sub(r"(?:\s*(?:<turn\|>|<eos>))+\s*$", "", text)
    text = re.sub(r"^\s*<\|turn>model\s*\n?", "", text)
    return text.strip()


def _parse_full_schema(text, mode="drugsafe"):
    cleaned = _clean_final_text(text)
    try:
        parsed = json.loads(cleaned)
    except Exception as exc:
        return False, str(exc), cleaned
    required_order = ["confidence", "defer_to_professional", "explanation", "flags", "citations"]
    flag_keys = {"severity", "description", "citation"}
    citation_keys = {"source", "text"}
    ok = (
        isinstance(parsed, dict)
        and list(parsed.keys()) == required_order
        and isinstance(parsed.get("confidence"), (int, float))
        and isinstance(parsed.get("defer_to_professional"), bool)
        and isinstance(parsed.get("explanation"), str)
        and isinstance(parsed.get("flags"), list)
        and all(isinstance(flag, dict) and flag_keys.issubset(flag) for flag in parsed.get("flags", []))
        and isinstance(parsed.get("citations"), list)
        and (mode != "drugsafe" or len(parsed.get("citations", [])) > 0)
        and all(isinstance(citation, dict) and citation_keys.issubset(citation) for citation in parsed.get("citations", []))
    )
    return ok, "" if ok else "schema/key-order/mode-aware-citations mismatch", cleaned


def _extract_calls(text):
    calls = []
    invalid = []
    for match in NATIVE_TC_RE.finditer(text):
        args = _parse_native_args(match.group(2))
        item = {"name": match.group(1), "args": args, "end": match.end(), "raw": match.group(0)}
        if args is None:
            invalid.append(item)
        else:
            calls.append(item)
    return calls, invalid


def _call_key(call):
    return (call["name"], json.dumps(call.get("args") or {}, sort_keys=True))


def _run_tool_loop(mode, user_msg, max_turns=4):
    messages = [
        {"role": "system", "content": _system_prompt(mode)},
        {"role": "user", "content": user_msg},
    ]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, tools=TOOL_DEFS_LIST)
    conversation = formatted
    seen_calls = set()
    trace = []

    for turn in range(1, max_turns + 1):
        raw = _gen_one(conversation, max_new_tokens=700, stop_for_tools=True)
        calls, invalid = _extract_calls(raw)
        if invalid:
            trace.append({"turn": turn, "raw": raw, "calls": [], "final_ok": None})
            return False, f"invalid tool call args: {invalid[:1]}", trace, raw
        if calls:
            repeated = [_call_key(call) for call in calls if _call_key(call) in seen_calls]
            if repeated:
                trace.append({"turn": turn, "raw": raw, "calls": [c["name"] for c in calls], "final_ok": None})
                return False, f"repeated tool call: {repeated[0]}", trace, raw
            for call in calls:
                seen_calls.add(_call_key(call))
            unknown = [call["name"] for call in calls if call["name"] not in ALLOWED_TOOL_NAMES]
            if unknown:
                trace.append({"turn": turn, "raw": raw, "calls": [c["name"] for c in calls], "final_ok": None})
                return False, f"unknown tool names: {unknown}", trace, raw
            last_end = max(call["end"] for call in calls)
            conversation += raw[:last_end]
            for call in calls:
                conversation += _format_tool_response(call["name"], _fake_tool_result(call["name"], call["args"]))
            conversation += "<turn|>\n<|turn>model\n"
            trace.append({"turn": turn, "raw": raw[:last_end], "calls": [c["name"] for c in calls], "final_ok": None})
            continue

        ok, detail, cleaned = _parse_full_schema(raw, mode=mode)
        trace.append({"turn": turn, "raw": raw, "calls": [], "final_ok": ok})
        if ok and "<|tool_response>" not in raw and "<|tool_call>" not in raw and "<|channel>" not in raw and "```" not in raw:
            return True, "final json ok", trace, raw
        return False, f"final invalid: {detail}", trace, raw

    return False, f"hit max_turns={max_turns}", trace, trace[-1]["raw"] if trace else ""


def _run_final_json(mode, user_msg):
    messages = [
        {"role": "system", "content": _system_prompt(mode)},
        {"role": "user", "content": user_msg},
    ]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    raw = _gen_one(formatted, max_new_tokens=700, stop_for_tools=False)
    ok, detail, cleaned = _parse_full_schema(raw, mode=mode)
    ok = ok and "<|tool_call>" not in raw and "<|tool_response>" not in raw and "<|channel>" not in raw and "```" not in raw
    return ok, detail, raw


failures = []
for i, (mode, expected, user_msg) in enumerate(SMOKE_TESTS, 1):
    print(f"\n[{i}/{len(SMOKE_TESTS)}] mode={mode} expected={expected}")
    print("prompt:", user_msg)
    if expected == "tool_loop":
        ok, detail, trace, raw = _run_tool_loop(mode, user_msg)
        print("result:", ok, detail)
        for step in trace:
            print(f"  turn {step['turn']}: calls={step['calls']} final_ok={step['final_ok']}")
            for line in step["raw"][:500].split("\n"):
                print(f"    | {line}")
        if not ok:
            failures.append((i, mode, expected, detail, raw[:700]))
    else:
        ok, detail, raw = _run_final_json(mode, user_msg)
        print("result:", ok, detail)
        for line in raw[:700].split("\n"):
            print(f"  | {line}")
        if not ok:
            failures.append((i, mode, expected, detail, raw[:700]))

print("\n" + "=" * 72)
print(" Verdict")
print("=" * 72)
print(f"passed: {len(SMOKE_TESTS) - len(failures)}/{len(SMOKE_TESTS)}")

if failures:
    print("\nFAILURES:")
    for idx, mode, expected, detail, preview in failures:
        print(f"- case {idx} mode={mode} expected={expected}: {detail}")
        print(f"  preview: {preview!r}")
    raise AssertionError("v4 smoke test failed; do not push merged checkpoint")

print("\nPASS: v4 smoke test passed. Proceed to push merged v4, then run held-out eval with the v4 eval kit.")


## Cell 8 - Push merged FP16 v4 to HF Hub

**This is the handoff to eval/export.** Once this cell completes, `eval_heldout.ipynb` should evaluate the v4 repo using the same output contract in its system prompt.

In [ ]:
from huggingface_hub import HfApi, create_repo

MERGED_REPO = "rescommons/aegis-sft-e4b-merged-v4"

create_repo(MERGED_REPO, private=True, exist_ok=True, token=os.environ["HF_TOKEN"])
HfApi().upload_folder(
    folder_path="/content/merged-v4",
    repo_id=MERGED_REPO,
    repo_type="model",
    token=os.environ["HF_TOKEN"],
)
print(f"Merged FP16 v4 pushed: https://huggingface.co/{MERGED_REPO}")
print("\nNext: open eval_heldout.ipynb or export_litertlm.ipynb in a fresh Colab runtime.")